In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(model=model, lr=10, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="armijo")
# opt = torch_numopt.GradientDescentLS(model=model, lr=1, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="wolfe")
# opt = torch_numopt.GradientDescentLS(model=model, lr=1, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="strong-wolfe")
# opt = torch_numopt.GradientDescentLS(model=model, lr=1, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="goldstein")

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.1337706297636032
epoch:  1, loss: 0.07328497618436813
epoch:  2, loss: 0.050919320434331894
epoch:  3, loss: 0.0418577715754509
epoch:  4, loss: 0.03817130625247955
epoch:  5, loss: 0.0366271510720253
epoch:  6, loss: 0.03593838959932327
epoch:  7, loss: 0.03560972586274147
epoch:  8, loss: 0.03545476496219635
epoch:  9, loss: 0.03541010990738869
epoch:  10, loss: 0.03524214029312134
epoch:  11, loss: 0.0351695641875267
epoch:  12, loss: 0.03513439744710922
epoch:  13, loss: 0.035132937133312225
epoch:  14, loss: 0.035052891820669174
epoch:  15, loss: 0.035014860332012177
epoch:  16, loss: 0.03499359264969826
epoch:  17, loss: 0.03493967279791832
epoch:  18, loss: 0.03489790856838226
epoch:  19, loss: 0.03487511724233627
epoch:  20, loss: 0.03483101353049278
epoch:  21, loss: 0.03478390350937843
epoch:  22, loss: 0.03475947678089142
epoch:  23, loss: 0.034726690500974655
epoch:  24, loss: 0.0346735417842865
epoch:  25, loss: 0.034646619111299515
epoch:  26, loss: 0.0

In [6]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7394834132515788
Test metrics:  R2 = 0.6808476277442648


In [8]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr=10,
    c1=1e-4,
    tau=0.1,
    line_search_method="interpolate",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.22337204217910767
epoch:  1, loss: 0.09315051138401031
epoch:  2, loss: 0.05143143609166145
epoch:  3, loss: 0.03799522668123245
epoch:  4, loss: 0.0355602391064167
epoch:  5, loss: 0.03544049337506294
epoch:  6, loss: 0.03541884943842888
epoch:  7, loss: 0.03540298342704773
epoch:  8, loss: 0.03539542108774185
epoch:  9, loss: 0.03538912907242775
epoch:  10, loss: 0.035387683659791946
epoch:  11, loss: 0.03537631407380104
epoch:  12, loss: 0.03537469357252121
epoch:  13, loss: 0.0353730209171772
epoch:  14, loss: 0.03537149727344513
epoch:  15, loss: 0.03536994010210037
epoch:  16, loss: 0.03536853566765785
epoch:  17, loss: 0.03536713495850563
epoch:  18, loss: 0.03536585345864296
epoch:  19, loss: 0.0353645421564579
epoch:  20, loss: 0.03536329045891762
epoch:  21, loss: 0.03536202013492584
epoch:  22, loss: 0.03536079451441765
epoch:  23, loss: 0.035359542816877365
epoch:  24, loss: 0.035358328372240067
epoch:  25, loss: 0.03535708412528038
epoch:  26, loss: 0.03

In [9]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7122529957651365
Test metrics:  R2 = 0.6571864470005637
